In [47]:
import sys
sys.path.insert(0, '..')

from oasis_dla import *

In [48]:
import sys
sys.path.insert(0, '../dla_compute/build')
import cudaq  # must be imported first to initialize CUDAQ runtime
import compute_dla as oasis_dla_cpp

In [49]:
import scipy as sp
import numpy as np
import sympy as smp
import scipy.stats as stats
import matplotlib.pyplot as plt


In [50]:
A = stats.unitary_group.rvs(2)
B = stats.unitary_group.rvs(2)

A, B

(array([[ 0.25142781+0.26283719j, -0.93146977-0.00804601j],
        [-0.57767984-0.73074391j, -0.36180542-0.03736534j]]),
 array([[-0.44809156+0.32008319j,  0.08245541+0.8306394j ],
        [ 0.44780959+0.70443401j, -0.55011607+0.02473066j]]))

In [51]:
n = 1_000

N = lambda ε: np.array([[ε, ε], [ε, ε]])

C = lambda ε: A + (1 / ε) * B
C(n)

array([[ 0.25097972+0.26315727j, -0.93138731-0.00721537j],
       [-0.57723204-0.73003948j, -0.36235554-0.03734061j]])

In [52]:
dla, dim = compute_dla(A, C(1000))

print(f"DLA: {dla}\nDimension: {dim}")

Iteration: 0, basis size of 2
Calculating [op[0], op[1]]
Added
[[-9.70805066e-04-0.00011966j -1.06139322e-04+0.00081006j]
 [ 9.37507091e-05-0.00081159j  9.70805066e-04+0.00011966j]]
Iteration: 1, basis size of 3
Calculating [op[0], op[1]]
Calculating [op[0], op[2]]
Calculating [op[1], op[2]]
Added
[[-0.00074712+0.00114561j -0.0021149 +0.00022634j]
 [ 0.00064561+0.00202662j  0.00074712-0.00114561j]]
Iteration: 2, basis size of 4
Calculating [op[0], op[1]]
Calculating [op[0], op[2]]
Calculating [op[0], op[3]]
Calculating [op[1], op[2]]
Calculating [op[1], op[3]]
Calculating [op[2], op[3]]
DLA: [array([[ 0.25142781+0.26283719j, -0.93146977-0.00804601j],
       [-0.57767984-0.73074391j, -0.36180542-0.03736534j]]), array([[ 0.25097972+0.26315727j, -0.93138731-0.00721537j],
       [-0.57723204-0.73003948j, -0.36235554-0.03734061j]]), array([[-9.70805066e-04-0.00011966j, -1.06139322e-04+0.00081006j],
       [ 9.37507091e-05-0.00081159j,  9.70805066e-04+0.00011966j]]), array([[-0.00074712+0.00

In [53]:
dla, dim = compute_dla(A, C(1_000_000))

print(f"DLA: {dla}\nDimension: {dim}")

Iteration: 0, basis size of 2
Calculating [op[0], op[1]]
Added
[[-9.70805066e-07-1.19664641e-07j -1.06139322e-07+8.10061842e-07j]
 [ 9.37507090e-08-8.11588903e-07j  9.70805066e-07+1.19664641e-07j]]
Iteration: 1, basis size of 3
Calculating [op[0], op[1]]
Calculating [op[0], op[2]]
Calculating [op[1], op[2]]
Added
[[-7.47118313e-07+1.14561194e-06j -2.11489627e-06+2.26343334e-07j]
 [ 6.45608531e-07+2.02662457e-06j  7.47118313e-07-1.14561194e-06j]]
Iteration: 2, basis size of 4
Calculating [op[0], op[1]]
Calculating [op[0], op[2]]
Calculating [op[0], op[3]]
Calculating [op[1], op[2]]
Calculating [op[1], op[3]]
DLA: [array([[ 0.25142781+0.26283719j, -0.93146977-0.00804601j],
       [-0.57767984-0.73074391j, -0.36180542-0.03736534j]]), array([[ 0.25142736+0.26283751j, -0.93146969-0.00804518j],
       [-0.5776794 -0.73074321j, -0.36180597-0.03736532j]]), array([[-9.70805066e-07-1.19664641e-07j, -1.06139322e-07+8.10061842e-07j],
       [ 9.37507090e-08-8.11588903e-07j,  9.70805066e-07+1.19664

In [54]:
dla, dim = compute_dla(A, C(1_000_000_000))

print(f"DLA: {dla}\nDimension: {dim}")

Iteration: 0, basis size of 2
DLA: [array([[ 0.25142781+0.26283719j, -0.93146977-0.00804601j],
       [-0.57767984-0.73074391j, -0.36180542-0.03736534j]]), array([[ 0.25142781+0.26283719j, -0.93146977-0.00804601j],
       [-0.57767984-0.73074391j, -0.36180542-0.03736534j]])]
Dimension: 2


In [55]:
oasis_dla_cpp.sample_bell(2, 1000)

{'00': 511, '11': 489}

Computing Group Orbits

$SL_2 \times SL_2$

-> $\mathfrak{sl}_2 \oplus \mathfrak{sl}_2$

In [71]:
# sl_2 basis

e = np.array([
    [0, 1],
    [0, 0]
])

h = np.array([
    [1, 0],
    [0, -1]
])

f = np.array([
    [0, 0],
    [1, 0]
])

I = np.eye(2)

sl2 = [e, h, f]

sl2_2 = [np.kron(I, X) for X in sl2] + [np.kron(X, I) for X in sl2]

def calc_orbit(algebra, vector):
    results = []
    # results.append(vector) # v + g.x

    for M in algebra:
        # print(M)
        action = M @ vector

        if np.allclose(action, 0):
            continue

        if results and not linear_indep(*results, action):
            continue

        results.append(action)

    if not results:
        return [], 0

    matrix = np.column_stack(results)
    rank = np.linalg.matrix_rank(matrix)

    return results, rank

calc_orbit(sl2_2, np.array([[1], [0], [0], [1]]))
# sl2_2

([array([[0.],
         [0.],
         [1.],
         [0.]]),
  array([[ 1.],
         [ 0.],
         [ 0.],
         [-1.]]),
  array([[0.],
         [1.],
         [0.],
         [0.]])],
 np.int64(3))

$\mathfrak{sl}_2 \oplus \mathfrak{sl}_2 \oplus \mathfrak{sl}_2$

In [67]:
sl2_3 = (
    [np.kron(np.kron(X, I), I) for X in sl2] +
    [np.kron(np.kron(I, X), I) for X in sl2] +
    [np.kron(np.kron(I, I), X) for X in sl2]
)

v = np.array([
    [1],
    [0],
    [0],
    [0],
    [0],
    [0],
    [0],
    [0],
])

# calc_orbit(sl2_3, v)


In [68]:
root2 = np.sqrt(2)
root3 = np.sqrt(3)

GHZ = np.array([
    [root2], # 000
    [0],
    [0],
    [0],
    [0],
    [0],
    [0],
    [root2], # 111
])

W = np.array([
    [0],     # 000
    [root3], # 001
    [root3], # 010
    [0],     # 011
    [root3],     # 100
    [0],
    [0],
    [0], # 111
])

calc_orbit(sl2_3, W)

([array([[0.        ],
         [1.73205081],
         [1.73205081],
         [0.        ],
         [1.73205081],
         [0.        ],
         [0.        ],
         [0.        ]]),
  array([[1.73205081],
         [0.        ],
         [0.        ],
         [0.        ],
         [0.        ],
         [0.        ],
         [0.        ],
         [0.        ]]),
  array([[ 0.        ],
         [ 1.73205081],
         [ 1.73205081],
         [ 0.        ],
         [-1.73205081],
         [ 0.        ],
         [ 0.        ],
         [ 0.        ]]),
  array([[0.        ],
         [0.        ],
         [0.        ],
         [0.        ],
         [0.        ],
         [1.73205081],
         [1.73205081],
         [0.        ]]),
  array([[ 0.        ],
         [ 1.73205081],
         [-1.73205081],
         [ 0.        ],
         [ 1.73205081],
         [ 0.        ],
         [ 0.        ],
         [ 0.        ]]),
  array([[0.        ],
         [0.        ],
        

In [69]:
calc_orbit(sl2_3, GHZ)

([array([[1.41421356],
         [0.        ],
         [0.        ],
         [0.        ],
         [0.        ],
         [0.        ],
         [0.        ],
         [1.41421356]]),
  array([[0.        ],
         [0.        ],
         [0.        ],
         [1.41421356],
         [0.        ],
         [0.        ],
         [0.        ],
         [0.        ]]),
  array([[ 1.41421356],
         [ 0.        ],
         [ 0.        ],
         [ 0.        ],
         [ 0.        ],
         [ 0.        ],
         [ 0.        ],
         [-1.41421356]]),
  array([[0.        ],
         [0.        ],
         [0.        ],
         [0.        ],
         [1.41421356],
         [0.        ],
         [0.        ],
         [0.        ]]),
  array([[0.        ],
         [0.        ],
         [0.        ],
         [0.        ],
         [0.        ],
         [1.41421356],
         [0.        ],
         [0.        ]]),
  array([[0.        ],
         [0.        ],
         [1.4142